<a href="https://colab.research.google.com/github/YukiIto0228/yuki-s/blob/main/Qwen3_TTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗣️ Qwen3-TTS Colab

## 📄 Description

This Colab notebook runs **Qwen3-TTS-12Hz-0.6B-Base** and **Qwen3-TTS-12Hz-1.7B-Base**, powerful **multilingual, low-latency text-to-speech (TTS)** models from the **Qwen3 TTS family**.
Designed with a **universal end-to-end architecture**, Qwen3-TTS delivers **high-fidelity**, **instruction-controllable**, and **real-time streaming** speech synthesis with strong robustness to noisy or complex text inputs.

**Capabilities:**
Multilingual TTS (10 Languages), Ultra-Low-Latency Streaming (≈97ms), Instruction-Based Voice Control, Rapid 3s Voice Cloning, High-Fidelity Speech Reconstruction

---

## How to use

* Modify text and instruction variables
* Run all following cells, upload reference audio if needed, and generate speech

---

## ⚙️ Model Highlights

* 🌍 **10-language support** – Chinese, English, Japanese, Korean, German, French, Russian, Portuguese, Spanish, Italian
* ⚡ **Extreme low-latency generation** – first audio packet emitted after a single character input
* 🧠 **Instruction-aware speech synthesis** – adaptive control over tone, emotion, prosody, and speaking rate
* 🧬 **3-second rapid voice cloning** – supported by both Base models
* 🏗 **End-to-end discrete LM architecture** – avoids cascading errors of traditional TTS pipelines

---

## 🧠 Model Details

* **Models Included:** Qwen3-TTS-12Hz-0.6B-Base, Qwen3-TTS-12Hz-1.7B-Base
* **Speech Tokenizer:** Qwen3-TTS-Tokenizer-12Hz
* **Architecture:** Discrete multi-codebook LM (non-DiT)
* **Streaming Support:** Yes (streaming & non-streaming in one model)
* **Latency:** As low as ~97 ms end-to-end
* **Use Cases:** Real-time assistants, voice agents, multilingual narration, TTS fine-tuning

---

## 🔗 Resources

* **Hugging Face (1.7B):** https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-Base  
* **Hugging Face (0.6B):** https://huggingface.co/Qwen/Qwen3-TTS-12Hz-0.6B-Base  
* **Official Blog:** https://qwen.ai/blog

---

## 🎙️ Explore More TTS Models

Looking for more cutting-edge voice models?
👉 Check out the full collection: [awesome-TTS-Colab](https://github.com/Troyanovsky/awesome-TTS-Colab)


## TTS/Voice Generation with Voice Cloning

In [ ]:
!pip -q install -U qwen-tts soundfile

# (Optional, recommended on GPU) FlashAttention 2 for lower memory + faster attention.
# If this fails (GPU not compatible / build issues), the notebook will still run without it.
try:
    import flash_attn  # noqa: F401
    print("flash-attn already installed.")
except Exception:
    !pip -q install -U flash-attn --no-build-isolation

In [ ]:
import torch
import os
import time
import soundfile as sf
from qwen_tts import Qwen3TTSModel
from google.colab import files

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"
LANGUAGE = "Japanese"

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
attn_impl = "flash_attention_2" if torch.cuda.is_available() else None

print(f"モデルの読み込みを開始します: {MODEL_ID}")
model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map=device,
    dtype=dtype,
    attn_implementation=attn_impl,
)
print("✅ モデルの読み込みが完了しました！")

In [ ]:
print("使い回す【参照音声】をアップロードしてください:")
uploaded_ref = files.upload()
ref_audio_path = next(iter(uploaded_ref.keys()))

# --------------------------------------------------
# 【重要】ここに参照音声の「書き起こし」を直接入力します
# --------------------------------------------------
ref_text = "ここに参照音声が喋っている内容（標準語や方言の書き起こし）を入力してください"

print(f"✅ 参照音声セット完了: {ref_audio_path}")
print(f"📝 書き起こしテキスト: {ref_text}")

In [ ]:
input_file = "marged-corpus.txt"
output_dir = "outputs"

os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(input_file):
    raise FileNotFoundError(f"エラー: {input_file} が見つかりません。左メニューからアップロードしてください。")

# 1. コーパスの読み込みと30件抽出
with open(input_file, "r", encoding="utf-8") as f:
    corpus_lines = [line.strip() for line in f.readlines() if line.strip()]

target_lines = corpus_lines[:30]
print(f"全{len(corpus_lines)}件中、上位{len(target_lines)}件の生成を開始します...")
print("-" * 50)

start_time = time.time()

# 2. 参照音声を固定して30回ループ
for i, input_text in enumerate(target_lines):
    current_index = i + 1
    out_clone_path = os.path.join(output_dir, f"qwen_tsugaru_{current_index:03d}.wav")

    try:
        # 毎回同じ ref_audio と ref_text を渡しつつ、input_text だけを変える
        wavs, sr = model.generate_voice_clone(
            text=input_text,
            language=LANGUAGE,
            ref_audio=ref_audio_path,
            ref_text=ref_text,
        )

        sf.write(out_clone_path, wavs[0], sr)
        print(f"[{current_index:03d}/030] 生成完了: {out_clone_path}")

    except Exception as e:
        print(f"[{current_index:03d}/030] ❌ エラー ({input_text[:10]}...): {e}")

end_time = time.time()
print("-" * 50)
print(f"🎉 30件のバッチ処理が完了しました！ (所要時間: {end_time - start_time:.2f}秒)")

In [ ]:
import shutil

zip_filename = 'qwen3_tsugaru_outputs'
shutil.make_archive(zip_filename, 'zip', output_dir)
files.download(f"{zip_filename}.zip")
print("✅ ダウンロードを開始しました。")